# Phase 2 — Feature Engineering

Demonstrates the graph-based (GNN-inspired) and temporal (TCN-inspired) feature engineering pipelines. See `docs/architecture.md` Sections 3.1/3.2.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd, networkx as nx
from src.data import io as data_io
from src.features.graph_features import attention_message_passing, neighbor_exposure_features
from src.features.temporal_features import build_dilated_features

nodes = data_io.load_raw('nodes')
edges = data_io.load_raw('edges')
ts = data_io.load_raw('port_timeseries')
print(nodes.shape, edges.shape, ts.shape)

(450, 18) (592, 3) (7300, 3)


## Graph feature engineering (attention message passing + neighbor exposure)

In [2]:
G = nx.Graph()
G.add_nodes_from(nodes.entity_id)
G.add_edges_from(list(zip(edges.source, edges.target)))
ids = nodes.entity_id.tolist()

feature_cols = ['otif_rate','ccc_days','debt_to_equity','current_ratio','ebitda_margin']
X = nodes[feature_cols].values
H = attention_message_passing(X, ids, G, emb_dim=8, n_layers=2, seed=7)
print("Attention embedding shape:", H.shape)

risk_proxy = nodes.set_index('entity_id')[['debt_to_equity','ccc_days','otif_rate']]
nbr_feats = neighbor_exposure_features(ids, G, risk_proxy, ['debt_to_equity','ccc_days','otif_rate'])
nbr_feats.head()

Attention embedding shape: (450, 8)


,entity_id,nbr1_debt_to_equity,nbr2_debt_to_equity,nbr1_ccc_days,nbr2_ccc_days,nbr1_otif_rate,nbr2_otif_rate
0,SUP_0000,0.98,0.652500,64.30,48.85,0.96660,0.940825
1,SUP_0001,0.73,0.642667,54.40,48.70,0.93635,0.920547
2,SUP_0002,0.20,0.532000,31.80,47.70,0.99000,0.943100
3,SUP_0003,0.42,0.721250,41.50,60.75,0.99000,0.919100
4,SUP_0004,0.98,0.603125,71.75,51.50,0.89025,0.946825


## Temporal feature engineering (dilated causal lags, TCN-inspired)

In [3]:
one_port_ts = ts[ts.port_id == ts.port_id.unique()[0]]
feats = build_dilated_features(one_port_ts, 'throughput_teu', dilations=[1,2,4,8,16,32])
print("Feature columns:", [c for c in feats.columns if c not in ('day','_y')])
feats.tail()

Feature columns: ['lag_1', 'lag_2', 'lag_4', 'lag_8', 'lag_16', 'lag_32', 'roll_mean_7', 'roll_mean_14', 'roll_mean_30', 'roll_std_7', 'dow_sin', 'dow_cos', 'annual_sin', 'annual_cos']


,lag_1,lag_2,lag_4,lag_8,lag_16,lag_32,roll_mean_7,roll_mean_14,roll_mean_30,roll_std_7,dow_sin,dow_cos,annual_sin,annual_cos,day,_y
360,104233.7,102557.1,93207.2,100329.2,101860.9,95233.2,98474.357143,98773.000000,98334.040000,4465.613744,0.433884,-0.900969,-0.090190,0.995925,360,100058.0
361,100058.0,104233.7,96065.5,102131.2,101040.6,93760.8,98178.185714,98898.007143,98408.003333,4246.010659,-0.433884,-0.900969,-0.073045,0.997329,361,101233.6
362,101233.6,100058.0,102557.1,97286.3,98307.9,97839.1,98742.085714,98903.807143,98487.233333,4368.177594,-0.974928,-0.222521,-0.055879,0.998438,362,97264.7
363,97264.7,101233.6,104233.7,93839.5,101152.4,98856.7,99231.400000,98869.171429,98548.810000,3893.524015,-0.781831,0.623490,-0.038696,0.999251,363,99947.6
364,99947.6,97264.7,100058.0,93207.2,97749.6,95417.4,100194.314286,99259.992857,98606.196667,2848.646020,0.000000,1.000000,-0.021501,0.999769,364,100871.8
